In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mstats
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel('../data/raw/E Commerce Dataset.xlsx', sheet_name='E Comm')
print(f"Shape awal: {df.shape}")
df.head()

Shape awal: (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [3]:
# CustomerID tidak diperlukan untuk modeling
df = df.drop(columns=['CustomerID'])
print(f"Shape setelah drop CustomerID: {df.shape}")

Shape setelah drop CustomerID: (5630, 19)


In [4]:
# Isi missing values numerik dengan median
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numerical_cols.remove('Churn')  # jangan impute target

for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"{col}: diisi dengan median = {median_val}")

print(f"\nMissing values tersisa: {df.isnull().sum().sum()}")

Tenure: diisi dengan median = 9.0
WarehouseToHome: diisi dengan median = 14.0
HourSpendOnApp: diisi dengan median = 3.0
OrderAmountHikeFromlastYear: diisi dengan median = 15.0
CouponUsed: diisi dengan median = 1.0
OrderCount: diisi dengan median = 2.0
DaySinceLastOrder: diisi dengan median = 3.0

Missing values tersisa: 0


In [5]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numerical_cols.remove('Churn')

for col in numerical_cols:
    df[col] = mstats.winsorize(df[col], limits=[0.05, 0.05])
    
print("Winsorization selesai!")
print(df[numerical_cols].describe().round(2))

Winsorization selesai!
        Tenure  CityTier  WarehouseToHome  HourSpendOnApp  \
count  5630.00   5630.00          5630.00         5630.00   
mean     10.01      1.65            15.41            2.94   
std       8.05      0.92             7.81            0.69   
min       0.00      1.00             6.00            2.00   
25%       3.00      1.00             9.00            2.00   
50%       9.00      1.00            14.00            3.00   
75%      15.00      3.00            20.00            3.00   
max      27.00      3.00            32.00            4.00   

       NumberOfDeviceRegistered  SatisfactionScore  NumberOfAddress  Complain  \
count                   5630.00            5630.00          5630.00   5630.00   
mean                       3.70               3.07             4.19      0.28   
std                        0.87               1.38             2.51      0.45   
min                        2.00               1.00             1.00      0.00   
25%                   

In [6]:
categorical_cols = ['PreferredLoginDevice', 'PreferredPaymentMode', 
                    'Gender', 'PreferedOrderCat', 'MaritalStatus']

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])
    print(f"{col}: {df[col].unique()}")

print(f"\nShape setelah encoding: {df.shape}")
df.head()

PreferredLoginDevice: [1 2 0]
PreferredPaymentMode: [4 6 0 2 5 1 3]
Gender: [0 1]
PreferedOrderCat: [2 3 4 5 0 1]
MaritalStatus: [2 0 1]

Shape setelah encoding: (5630, 19)


,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,1,4.0,1,3,6.0,4,0,3.0,3,2,2,2,9,1,11.0,1.0,1.0,5.0,159.93
1,1,9.0,2,1,8.0,6,1,3.0,4,3,3,2,7,1,15.0,0.0,1.0,0.0,123.02
2,1,9.0,2,1,30.0,4,1,2.0,4,3,3,2,6,1,14.0,0.0,1.0,3.0,123.02
3,1,0.0,2,3,15.0,4,1,2.0,4,2,5,2,8,0,23.0,0.0,1.0,3.0,134.07
4,1,0.0,2,1,12.0,0,1,3.0,3,3,5,2,3,0,11.0,1.0,1.0,3.0,129.60


In [7]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"\nDistribusi target sebelum SMOTE:")
print(y.value_counts())

Shape X: (5630, 18)
Shape y: (5630,)

Distribusi target sebelum SMOTE:
Churn
0    4682
1     948
Name: count, dtype: int64


In [8]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"Shape X setelah SMOTE: {X_resampled.shape}")
print(f"\nDistribusi target setelah SMOTE:")
print(pd.Series(y_resampled).value_counts())

Shape X setelah SMOTE: (9364, 18)

Distribusi target setelah SMOTE:
Churn
1    4682
0    4682
Name: count, dtype: int64


In [9]:
df_processed = pd.DataFrame(X_resampled, columns=X.columns)
df_processed['Churn'] = y_resampled

df_processed.to_csv('../data/processed/data_preprocessed.csv', index=False)
print(f"Data tersimpan! Shape final: {df_processed.shape}")

Data tersimpan! Shape final: (9364, 19)


In [10]:
print("=" * 50)
print("SUMMARY PREPROCESSING")
print("=" * 50)
print(f"""
✅ SELESAI
- Shape awal         : (5630, 20)
- Shape final        : (9364, 19)
- Missing values     : 0
- Outlier            : Winsorized (5% tiap sisi)
- Encoding           : LabelEncoder (5 kolom)
- Class balance      : SMOTE → 4682 vs 4682
- File tersimpan     : data/processed/data_preprocessed.csv
""")

SUMMARY PREPROCESSING

✅ SELESAI
- Shape awal         : (5630, 20)
- Shape final        : (9364, 19)
- Missing values     : 0
- Outlier            : Winsorized (5% tiap sisi)
- Encoding           : LabelEncoder (5 kolom)
- Class balance      : SMOTE → 4682 vs 4682
- File tersimpan     : data/processed/data_preprocessed.csv

